# Análisis de resultados

Lee **solo** de `results/tables/` y `results/runs/`. Nunca recalcula el experimento: si falta algo, se corre el script correspondiente.

In [ ]:
import pandas as pd

from anonlogs.config import REPO_ROOT

tables = REPO_ROOT / "results" / "tables"

## Carga de runs

Lee los `runs.csv` de `results/sessions/` (una fila por run). No recalcula nada: solo consolida lo ya ejecutado.

In [ ]:
sessions = REPO_ROOT / "results" / "sessions"
run_files = sorted(sessions.glob("*/runs.csv"))
runs = (pd.concat([pd.read_csv(f) for f in run_files], ignore_index=True)
        if run_files else pd.DataFrame())
ok = runs[runs["status"] == "ok"].copy()
for col in ("recall", "f1"):
    ok[col] = pd.to_numeric(ok[col], errors="coerce")
ok = ok.dropna(subset=["recall", "f1"], how="all")
len(ok)

## Comparativa completa (tratamiento × detector)

Reproduce el estilo de `comparacion_*.csv`, **conservando seudonimizada**. Media de recall y F1 por tratamiento y detector, sobre todos los escenarios disponibles.

In [ ]:
def tabla_detector(df):
    """Pivote tratamiento × detector con media de recall y f1 (estilo comparacion_*.csv)."""
    piv = df.pivot_table(index="tratamiento", columns="detector",
                         values=["recall", "f1"], aggfunc="mean")
    piv.columns = [f"{det}__{met}" for met, det in piv.columns]
    return piv.sort_index(axis=1).round(3)

orden = ["original", "seudonimizada", "ip_generalizada", "suprimida"]
completa = tabla_detector(ok)
completa = completa.reindex([t for t in orden if t in completa.index])
completa

## Comparativa enfocada: Generalización vs Supresión por escenario

La comparativa completa promedia sobre todos los escenarios y **oculta el efecto del número de IP de origen**, que es justo donde `ip_generalizada` (/24) y `suprimida` divergen: con **1 IP** ambos colapsan la entidad por igual; con **N IP**, la supresión funde todas las IP en un único token mientras que /24 solo agrupa las del mismo bloque (en el lab, `attacker_1`/`attacker_2` comparten /24 y `attacker_3` no). Por eso se desglosa por escenario, separando los de 1 IP (`concentrado`, `multi_cuenta`) de los de N IP (`distribuido`, `spraying`).

In [ ]:
# dataset -> (escenario, n_ips). Los escenarios aún sin capturar simplemente no aparecen.
ESCENARIOS = {
    "lab_concentrado":  ("concentrado",  "1 IP"),
    "lab_multi_cuenta": ("multi_cuenta", "1 IP"),
    "lab_distribuido":  ("distribuido",  "N IP"),
    "lab_spraying":     ("spraying",     "N IP"),
}

foco = ok[ok["tratamiento"].isin(["ip_generalizada", "suprimida"])].copy()
foco["escenario"] = foco["dataset"].map(lambda d: ESCENARIOS.get(d, (d, "?"))[0])
foco["n_ips"]     = foco["dataset"].map(lambda d: ESCENARIOS.get(d, (d, "?"))[1])

gen_vs_sup = foco.pivot_table(index=["n_ips", "escenario", "tratamiento"],
                              columns="detector", values=["recall", "f1"], aggfunc="mean")
gen_vs_sup.columns = [f"{det}__{met}" for met, det in gen_vs_sup.columns]
gen_vs_sup = gen_vs_sup.sort_index(axis=1).round(3).sort_index()
gen_vs_sup

Se guarda en `results/tables/comparacion_gen_vs_sup.csv` (liviano, se commitea como el resto de `results/tables/`).

In [ ]:
out = tables / "comparacion_gen_vs_sup.csv"
gen_vs_sup.to_csv(out)
print("escrito:", out)